In [1]:
import sys
import os
current_path = os.path.abspath('')
data_path = '/'.join(current_path.split('/')[:-2]) + '/data/'
log_path = '/'.join(current_path.split('/')[:-1]) + '/logs_annotated/'
script_path = '/'.join(current_path.split('/')[:-1]) + '/scripts/'
sys.path.append(script_path)
sys.path.append('/home/fkunneman/.local/lib/python3.10/site-packages/')
sys.path.append('/usr/lib/python3/dist-packages/')

In [2]:
import torch
import gc
import importlib
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_community.vectorstores import Chroma
import chromadb
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings

import instruction_agent
import generate_response

/home/fkunneman/.local/lib/python3.10/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [3]:
### initialize huggingface
with open('/home/fkunneman/hf.txt') as fin:
    hf = fin.read().strip()
os.environ["HF_TOKEN"] = hf
torch.cuda.empty_cache()
gc.collect()

0

In [ ]:
# ### Load judge llm
# tokenizer_judge = AutoTokenizer.from_pretrained('BramVanroy/fietje-2-instruct')
# tokenizer_judge.pad_token = tokenizer_judge.eos_token # Ensure padding token is set
# judgemodel = AutoModelForCausalLM.from_pretrained('BramVanroy/fietje-2-instruct', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")

# #tokenizer_judge = AutoTokenizer.from_pretrained('mistralai/Mistral-Small-3.1-24B-Instruct-2503')
# #tokenizer_judge.pad_token = tokenizer_judge.eos_token # Ensure padding token is set
# #judgemodel = AutoModelForCausalLM.from_pretrained('mistralai/Mistral-Small-3.1-24B-Instruct-2503', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")
# # set judge pipeline
# pipe = pipeline(
#     "text-generation",
#     model=judgemodel,
#     tokenizer=tokenizer_judge,
#     return_full_text=False,
#     #max_new_tokens=250, # Limit the number of generated tokens for concise responses
#     do_sample=False, # Enable sampling for more varied responses
#     temperature=0.3 # Control creativity
# )
# llm = HuggingFacePipeline(pipeline=pipe)

In [4]:
### Load models for chatbot and rag in memory
with open('/home/fkunneman/hf.txt') as fin:
    hf = fin.read().strip()
os.environ["HF_TOKEN"] = hf
torch.cuda.empty_cache()
gc.collect()
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained('BramVanroy/GEITje-7B-ultra', use_fast=True)
tokenizer.pad_token = tokenizer.eos_token # Ensure padding token is set
# Initialize language model
chatmodel = AutoModelForCausalLM.from_pretrained('BramVanroy/GEITje-7B-ultra', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")
#chatmodel = AutoModelForCausalLM.from_pretrained('robinsmits/Qwen1.5-7B-Dutch-Chat', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")     
# set pipeline
pipe = pipeline(
    "text-generation",
    model=chatmodel,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=250, # Limit the number of generated tokens for concise responses
    do_sample=True, # Enable sampling for more varied responses
    temperature=0.3 # Control creativity
)
llm = HuggingFacePipeline(pipeline=pipe)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [32]:
chroma_client = chromadb.Client()
#embedding_model = SentenceTransformer('jegormeister/bert-base-dutch-cased')
collection = chroma_client.create_collection(
    name="instruction_db6",
    configuration={
        "hnsw": {
            "space": "cosine",
            "ef_construction": 30
        }
    }
)

In [33]:
instructions_travel = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_ov_stripped.csv'
instructions_passport = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_paspoort_stripped_v2.csv'
pat = data_path + 'opslag_inclusieve_spraakassistent_project/patterns_v2.csv'
qa_travel = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_ov_v5.csv'
qa_passport = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_paspoort.csv'
nav = data_path + 'opslag_inclusieve_spraakassistent_project/navigation.csv'


In [37]:
importlib.reload(instruction_agent)
assistant = instruction_agent.InstructAgent(llm,'travel')
assistant.prepare_patterns(pat)
assistant.prepare_instructions(instructions_travel,'travel')
assistant.prepare_instructions(instructions_passport,'passport')
assistant.setup_rag(collection,[['qa','travel',qa_travel],['qa','passport',qa_passport],['nav',nav]])
assistant.initiate()

In [38]:
# initialize response generator
importlib.reload(sys.modules['generate_response'])
from generate_response import Turns, ResponseGenerator

turns = Turns()
#turns.read_log(log_path + 'P9 conversatie paspoort annotated.csv')
turns.read_log(log_path + 'P7 conversatie reis2 annotated.csv')

rgen = ResponseGenerator(assistant)
for i,turn in enumerate(turns.turns):
    output, info = rgen.generate_response(turn,threshold=0.30,sp=2)
    print('TURN',i,':',turn['asr'],',',output,'\n', info)

TURN 0 :  ik wil starta. , Super! De eerste stap: onder de woorden 'Waar wil je heen?' staat een tekstvakje. Tik daar het station in waar vandaan je wil vertrekken. 
 ['Ik wil starten, Ik wil starten, Ik wil starten, Ik wil beginnen, Ik wil beginnen, Ik wil beginnen, Ik kan starten, Ik kan starten, Ik kan starten, Ik kan beginnen', 0.16, 76, 'travel', '1', 'nav', 'start', 0]
TURN 1 :  volgende... , Er staat nog een tweede tekstvakje, naast het woord 'Naar'. Vul daar het station in waar je naartoe wil. 
 ['Volgende, Volgende, Volgende, Volgende stap, Volgende stap, Volgende stap, Vorige, Vorige, Vorige, Einde gesprek', 0.05, 70, 'travel', '2', 'nav', 'next step', 1]


Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt took 2.943017094001334
TURN 2 :  volgende. ik wil weten welk perron. de trein naar utrecht. ,  Goed dat je het vraagt! Als je naar het station 'Utrecht' wilt, kijk dan onder het vakje 'Naar' voor het perronnummer. Dat is het nummer waar je trein zal vertrekken.
 
 ['Ik weet het niet, Ik weet het niet, Ik weet het niet, Ik snap het niet., Ik snap het niet., Ik snap het niet., Ik kan starten, Ik kan starten, Ik kan starten, Ik wil starten', 0.4, 27, 'travel', '2', 'nav', 'clarify', 1]
TURN 3 :  amsterdam centraal , Oké vul dat in bij ‘Naar’.  
 ["Amsterdam Centraal, Amsterdam Centraal, Amsterdam Centraal, Utrecht Centraal, Utrecht Centraal, Utrecht Centraal, Waar is het veld 'Van'?, Waar is het veld 'Van'?, Waar is het veld 'Van'?, Waar is 'Van'?", 0.0, 86, 'travel', '2', 'travel', 'Oké vul dat in bij ‘Naar’. ', 1]


Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt took 2.8514260579995607
TURN 4 :  ik zou het niet zeggen. ,  Geen probleem, als je het niet zeker weet, kun je gewoon de naam van het station typen. De website zal je dan helpen met de juiste spelling of het zoeken naar het station.

 
 ['Ik weet het niet, Ik weet het niet, Ik weet het niet, Ik snap het niet., Ik snap het niet., Ik snap het niet., Wat zeg je?, Wat zeg je?, Wat zeg je?, Kun je dat opnieuw zeggen?', 0.15, 55, 'travel', '2', 'nav', 'clarify', 1]
TURN 5 :  volgende... , Als je op een latere dag wil reizen, klik dan op de knop 'datum' en kies de dag. 
 ['Volgende, Volgende, Volgende, Volgende stap, Volgende stap, Volgende stap, Vorige, Vorige, Vorige, Einde gesprek', 0.05, 70, 'travel', '3', 'nav', 'next step', 2]
TURN 6 :  volgende. , Klik op de knop 'tijd'. Kies hoe laat je wilt vertrekken. 
 ['Volgende, Volgende, Volgende, Volgende stap, Volgende stap, Volgende stap, Vorige, Vorige, Vorige, Einde', 0.02, 78, 'travel', '4', 'nav', 'next step', 3]
TURN 7 :  volgende

Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt took 1.32700581500103
TURN 9 :  bedankt! ,  Graag gedaan! Als je nog meer vragen hebt, help ik je graag. Veel succes met je reis! 
 ["dankje, dankje, dankje, Waar staat 'tijd'?, Waar staat 'tijd'?, Waar staat 'tijd'?, Waar is het veld 'Naar'?, Waar is het veld 'Naar'?, Waar is het veld 'Naar'?, Waar is het veld 'Van'?", 0.61, 53, 'travel', 'e', 'travel', ' Graag gedaan!', 4]
